# 14_factor_eda — Feature / Factor IC Analysis

日本株予測システム向け 特徴量（ファクター）の IC 分析ノートブック。  
Information Coefficient (IC)・因子相関・デケイ・ターンオーバーを網羅的に分析する。

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy.stats import spearmanr, pearsonr
from statsmodels.stats.outliers_influence import variance_inflation_factor

try:
    import japanize_matplotlib
    print('japanize_matplotlib loaded')
except ImportError:
    print('japanize_matplotlib not available — using default font')

%matplotlib inline

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
print('Setup complete.')

## 1. データ読み込み

In [ ]:
SYMBOLS = [
    '7203.T', '6758.T', '8306.T', '9432.T', '9984.T', '6861.T', '8316.T',
    '4063.T', '6367.T', '7974.T', '8035.T', '6954.T', '4519.T', '9433.T',
    '3382.T', '4502.T', '7751.T', '6098.T', '8766.T', '6501.T', '4568.T',
    '7267.T', '6702.T', '6503.T', '5108.T'
]

DATA_PATH = Path('../../data/equity_jp_ohlcv.parquet')

def make_dummy_data(symbols=SYMBOLS, n_days=1500, start='2020-01-01', seed=42):
    """Generate realistic dummy OHLCV data via random walk."""
    rng = np.random.default_rng(seed)
    trading_dates = pd.bdate_range(start=start, periods=n_days)
    records = []
    for sym in symbols:
        price = rng.uniform(1000, 10000)
        prices = [price]
        for _ in range(n_days - 1):
            price = price * np.exp(rng.normal(0.0002, 0.018))
            prices.append(price)
        closes = np.array(prices)
        highs  = closes * (1 + rng.uniform(0, 0.03, n_days))
        lows   = closes * (1 - rng.uniform(0, 0.03, n_days))
        opens  = closes * (1 + rng.normal(0, 0.008, n_days))
        vols   = rng.lognormal(mean=np.log(1e6), sigma=0.8, size=n_days)
        for i, dt in enumerate(trading_dates):
            records.append({
                'date': dt,
                'symbol': sym,
                'open': opens[i],
                'high': highs[i],
                'low': lows[i],
                'close': closes[i],
                'volume': vols[i]
            })
    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'])
    return df

try:
    df = pd.read_parquet(DATA_PATH)
    df['date'] = pd.to_datetime(df['date'])
    # Filter to known symbols if column exists
    if 'symbol' in df.columns:
        df = df[df['symbol'].isin(SYMBOLS)].copy()
    print('Loaded from parquet.')
except FileNotFoundError:
    print(f'Parquet not found at {DATA_PATH.resolve()} — generating dummy data.')
    df = make_dummy_data()

df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

print(f'Shape       : {df.shape}')
print(f'Date range  : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Symbols ({len(df["symbol"].unique())}): {sorted(df["symbol"].unique())}')
df.head(3)

## 2. 特徴量計算

In [ ]:
# ── Helper: apply per-symbol transform ────────────────────────────────────
def sym_apply(df, col, func):
    return df.groupby('symbol', group_keys=False)[col].apply(func)

# ── Returns ────────────────────────────────────────────────────────────────
df['ret_1d']    = sym_apply(df, 'close', lambda s: s.pct_change(1))
df['ret_5d']    = sym_apply(df, 'close', lambda s: s.pct_change(5))
df['ret_20d']   = sym_apply(df, 'close', lambda s: s.pct_change(20))
df['logret_1d'] = sym_apply(df, 'close', lambda s: np.log(s / s.shift(1)))
df['logret_5d'] = sym_apply(df, 'close', lambda s: np.log(s / s.shift(5)))
df['logret_20d']= sym_apply(df, 'close', lambda s: np.log(s / s.shift(20)))

# ── Forward return label ───────────────────────────────────────────────────
df['fwd_ret_5d'] = sym_apply(df, 'close', lambda s: np.log(s.shift(-5) / s))

# ── SMA / EMA ─────────────────────────────────────────────────────────────
for w in [5, 10, 20, 60]:
    df[f'sma_{w}'] = sym_apply(df, 'close', lambda s, w=w: s.rolling(w).mean())
for w in [5, 20]:
    df[f'ema_{w}'] = sym_apply(df, 'close', lambda s, w=w: s.ewm(span=w, adjust=False).mean())

for w in [5, 10, 20, 60]:
    df[f'price_sma{w}_ratio'] = df['close'] / df[f'sma_{w}']

# ── RSI (exponential method) ───────────────────────────────────────────────
def compute_rsi(series, w):
    delta    = series.diff()
    gain     = delta.where(delta > 0, 0.0)
    loss     = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(com=w - 1, min_periods=w).mean()
    avg_loss = loss.ewm(com=w - 1, min_periods=w).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

df['rsi_14'] = sym_apply(df, 'close', lambda s: compute_rsi(s, 14))
df['rsi_28'] = sym_apply(df, 'close', lambda s: compute_rsi(s, 28))

# ── MACD ──────────────────────────────────────────────────────────────────
ema12 = sym_apply(df, 'close', lambda s: s.ewm(span=12, adjust=False).mean())
ema26 = sym_apply(df, 'close', lambda s: s.ewm(span=26, adjust=False).mean())
df['macd']        = ema12 - ema26
df['macd_signal'] = df.groupby('symbol', group_keys=False)['macd'].apply(
    lambda s: s.ewm(span=9, adjust=False).mean()
)
df['macd_hist'] = df['macd'] - df['macd_signal']

# ── Bollinger Bands ────────────────────────────────────────────────────────
df['bb_mid']   = sym_apply(df, 'close', lambda s: s.rolling(20).mean())
df['bb_std']   = sym_apply(df, 'close', lambda s: s.rolling(20).std())
df['bb_pct']   = (df['close'] - (df['bb_mid'] - 2 * df['bb_std'])) / (4 * df['bb_std'])
df['bb_width'] = (4 * df['bb_std']) / df['bb_mid']

# ── Volume features ────────────────────────────────────────────────────────
df['vol_chg_1d']        = sym_apply(df, 'volume', lambda s: s.pct_change(1))
df['vol_sma_ratio_5d']  = sym_apply(df, 'volume', lambda s: s / s.rolling(5).mean())
df['volume_zscore_20d'] = sym_apply(df, 'volume',
    lambda s: (s - s.rolling(20).mean()) / s.rolling(20).std())

# ── Volatility ─────────────────────────────────────────────────────────────
df['vol_5d']  = sym_apply(df, 'logret_1d', lambda s: s.rolling(5).std()  * np.sqrt(252))
df['vol_20d'] = sym_apply(df, 'logret_1d', lambda s: s.rolling(20).std() * np.sqrt(252))

print('Raw features computed.')

In [ ]:
# ── Cross-sectional rank features ──────────────────────────────────────────
CS_RANK_COLS = [
    'ret_1d', 'ret_5d', 'ret_20d',
    'logret_1d', 'logret_5d', 'logret_20d',
    'rsi_14', 'rsi_28',
    'macd', 'macd_hist',
    'vol_20d', 'vol_chg_1d', 'vol_sma_ratio_5d', 'volume_zscore_20d',
    'price_sma5_ratio', 'price_sma20_ratio',
    'bb_pct', 'bb_width'
]

for col in CS_RANK_COLS:
    df[f'{col}_csrank'] = df.groupby('date')[col].transform(
        lambda s: s.rank(pct=True) - 0.5
    )

# ── Define FEATURES list ───────────────────────────────────────────────────
EXCLUDE = {'symbol', 'date', 'open', 'high', 'low', 'close', 'volume',
           'bb_mid', 'bb_std', 'macd_signal'}  # intermediate cols not features
EXCLUDE.update({c for c in df.columns if c.startswith('fwd_ret')})
EXCLUDE.update({f'sma_{w}' for w in [5, 10, 20, 60]})
EXCLUDE.update({f'ema_{w}' for w in [5, 20]})

FEATURES = [c for c in df.columns if c not in EXCLUDE]

print(f'Number of features: {len(FEATURES)}')
print('Features:', FEATURES)
df[['date', 'symbol'] + FEATURES[:6] + ['fwd_ret_5d']].head(3)

## 3. IC 計算 (Information Coefficient)

In [ ]:
def monthly_ic(df, feature, target='fwd_ret_5d'):
    """Compute monthly Spearman IC between feature and target."""
    tmp = df[['date', feature, target]].dropna()
    tmp['period'] = tmp['date'].dt.to_period('M')
    ic_series = {}
    for period, grp in tmp.groupby('period'):
        if len(grp) < 5:
            continue
        rho, _ = spearmanr(grp[feature], grp[target])
        ic_series[period] = rho
    return pd.Series(ic_series)

ic_records = []
ic_ts_dict = {}  # feature -> monthly IC series

for feat in FEATURES:
    ic_s = monthly_ic(df, feat)
    if ic_s.isna().all() or len(ic_s) == 0:
        continue
    ic_ts_dict[feat] = ic_s
    mean_ic = ic_s.mean()
    std_ic  = ic_s.std()
    n       = len(ic_s)
    t_stat  = mean_ic / std_ic * np.sqrt(n) if std_ic > 0 else np.nan
    ir      = mean_ic / std_ic if std_ic > 0 else np.nan
    pos_ratio = (ic_s > 0).mean() * 100
    ic_records.append(dict(
        feature=feat,
        mean_IC=mean_ic,
        std_IC=std_ic,
        t_stat=t_stat,
        IR=ir,
        IC_pos_ratio=pos_ratio,
        N=n
    ))

ic_summary = pd.DataFrame(ic_records).set_index('feature')
ic_summary['abs_mean_IC'] = ic_summary['mean_IC'].abs()
ic_summary = ic_summary.sort_values('abs_mean_IC', ascending=False)

print(f'Features with IC computed: {len(ic_summary)}')
ic_summary.head(10)

### 3a. IC 時系列プロット（上位10特徴量）

In [ ]:
top10 = ic_summary.head(10).index.tolist()

fig, axes = plt.subplots(5, 2, figsize=(16, 18))
axes = axes.flatten()

for i, feat in enumerate(top10):
    ax = axes[i]
    ic_s = ic_ts_dict[feat]
    ic_s_float = ic_s.copy()
    ic_s_float.index = ic_s_float.index.to_timestamp()
    ax.bar(ic_s_float.index, ic_s_float.values,
           color=['steelblue' if v >= 0 else 'tomato' for v in ic_s_float.values],
           width=20, alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axhline(ic_s_float.mean(), color='navy', linewidth=1.2, linestyle='--',
               label=f'mean={ic_s_float.mean():.3f}')
    ax.set_title(f'{feat}\nIC mean={ic_s_float.mean():.3f}, IR={ic_summary.loc[feat,"IR"]:.2f}',
                 fontsize=9)
    ax.legend(fontsize=7)
    ax.tick_params(axis='x', labelrotation=30, labelsize=7)

plt.suptitle('Monthly Spearman IC — Top 10 Features (vs fwd_ret_5d)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 3b. Mean IC テーブル（上位20特徴量）

In [ ]:
display_cols = ['mean_IC', 'std_IC', 't_stat', 'IR', 'IC_pos_ratio', 'N']
top20_tbl = ic_summary[display_cols].head(20)
top20_tbl.style \
    .format({'mean_IC': '{:.4f}', 'std_IC': '{:.4f}', 't_stat': '{:.2f}',
             'IR': '{:.2f}', 'IC_pos_ratio': '{:.1f}%', 'N': '{:.0f}'}) \
    .background_gradient(subset=['mean_IC'], cmap='RdBu', vmin=-0.1, vmax=0.1) \
    .background_gradient(subset=['IR'], cmap='coolwarm', vmin=-2, vmax=2) \
    .set_caption('Top 20 Features by |mean IC| — Spearman IC vs fwd_ret_5d')

### 3c. Mean IC 棒グラフ（上位20特徴量）

In [ ]:
top20 = ic_summary.head(20)
colors = ['steelblue' if v >= 0 else 'tomato' for v in top20['mean_IC']]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top20.index[::-1], top20['mean_IC'][::-1], color=colors[::-1], alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean Spearman IC (monthly)')
ax.set_title('Mean IC — Top 20 Features (vs fwd_ret_5d)')

for bar, val in zip(bars, top20['mean_IC'][::-1]):
    ax.text(val + (0.001 if val >= 0 else -0.001), bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.show()

## 4. IC 安定性分析

### 4a. ローリング12ヶ月 IC（上位5特徴量）

In [ ]:
top5 = ic_summary.head(5).index.tolist()

fig, ax = plt.subplots(figsize=(14, 5))

for feat in top5:
    ic_s = ic_ts_dict[feat].copy()
    ic_s.index = ic_s.index.to_timestamp()
    rolling_ic = ic_s.rolling(12, min_periods=6).mean()
    ax.plot(rolling_ic.index, rolling_ic.values, label=feat, linewidth=1.5)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Rolling 12-Month Mean IC — Top 5 Features')
ax.set_xlabel('Date')
ax.set_ylabel('Rolling Mean IC')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

### 4b. IC Positive Ratio — 上位5 / 下位5

In [ ]:
pos_ratio_sorted = ic_summary[['IC_pos_ratio', 'mean_IC']].sort_values('IC_pos_ratio', ascending=False)

print('=== IC Positive Ratio — Top 5 ===')
print(pos_ratio_sorted.head(5).to_string())
print()
print('=== IC Positive Ratio — Bottom 5 ===')
print(pos_ratio_sorted.tail(5).to_string())

### 4c. IC Heatmap: 特徴量 × 年

In [ ]:
top15 = ic_summary.head(15).index.tolist()

yearly_ic = {}
for feat in top15:
    ic_s = ic_ts_dict[feat].copy()
    ic_s.index = ic_s.index.to_timestamp()
    yearly_ic[feat] = ic_s.groupby(ic_s.index.year).mean()

heatmap_df = pd.DataFrame(yearly_ic).T  # features x years

fig, ax = plt.subplots(figsize=(max(8, len(heatmap_df.columns) * 1.2), len(top15) * 0.55 + 1))
sns.heatmap(
    heatmap_df,
    cmap='RdBu', center=0, annot=True, fmt='.3f',
    linewidths=0.4, ax=ax, annot_kws={'size': 8},
    cbar_kws={'label': 'Mean IC'}
)
ax.set_title('Mean IC by Year — Top 15 Features', fontsize=12)
ax.set_xlabel('Year')
ax.set_ylabel('Feature')
ax.tick_params(axis='x', labelrotation=0)
ax.tick_params(axis='y', labelrotation=0, labelsize=8)
plt.tight_layout()
plt.show()

## 5. 因子相関行列

### 5a. Pearson 相関行列（ヒートマップ）

In [ ]:
feat_df = df[FEATURES].dropna(how='all')

# Sample if large
if len(feat_df) > 50_000:
    feat_df_sample = feat_df.sample(50_000, random_state=42)
else:
    feat_df_sample = feat_df.copy()

pearson_corr = feat_df_sample.corr(method='pearson')

# Mask upper triangle
mask = np.triu(np.ones_like(pearson_corr, dtype=bool), k=1)

n_feat = len(FEATURES)
fig_size = max(12, n_feat * 0.45)
fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))
sns.heatmap(
    pearson_corr, mask=mask,
    cmap='RdBu', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.2,
    ax=ax, cbar_kws={'shrink': 0.6},
    annot=(n_feat <= 20), fmt='.2f', annot_kws={'size': 7}
)
ax.set_title('Pearson Correlation Matrix — All Features', fontsize=12)
ax.tick_params(axis='x', labelrotation=90, labelsize=7)
ax.tick_params(axis='y', labelrotation=0,  labelsize=7)
plt.tight_layout()
plt.show()

### 5b. Spearman 相関行列（サンプル5000行）

In [ ]:
sample_5k = feat_df.dropna().sample(min(5000, len(feat_df.dropna())), random_state=42)
spearman_corr = sample_5k.corr(method='spearman')

mask2 = np.triu(np.ones_like(spearman_corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))
sns.heatmap(
    spearman_corr, mask=mask2,
    cmap='RdBu', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.2,
    ax=ax, cbar_kws={'shrink': 0.6},
    annot=(n_feat <= 20), fmt='.2f', annot_kws={'size': 7}
)
ax.set_title('Spearman Correlation Matrix — All Features (sample n=5000)', fontsize=12)
ax.tick_params(axis='x', labelrotation=90, labelsize=7)
ax.tick_params(axis='y', labelrotation=0,  labelsize=7)
plt.tight_layout()
plt.show()

### 5c. 高相関ペア (Pearson |ρ| > 0.7)

In [ ]:
high_corr_pairs = []
corr_vals = pearson_corr.values
feat_list = pearson_corr.columns.tolist()

for i, j in itertools.combinations(range(len(feat_list)), 2):
    rho = corr_vals[i, j]
    if abs(rho) > 0.7:
        high_corr_pairs.append({'feature_1': feat_list[i], 'feature_2': feat_list[j], 'pearson_rho': rho})

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('pearson_rho', key=abs, ascending=False)
print(f'High-correlation pairs (|ρ| > 0.7): {len(high_corr_df)}')
high_corr_df.style.format({'pearson_rho': '{:.4f}'}).background_gradient(subset=['pearson_rho'], cmap='RdBu', vmin=-1, vmax=1)

### 5d. VIF (Variance Inflation Factor)

In [ ]:
vif_data = feat_df.dropna()
if len(vif_data) > 5000:
    vif_data = vif_data.sample(5000, random_state=42)

# Drop zero-variance columns
vif_data = vif_data.loc[:, vif_data.std() > 1e-8]
vif_cols = vif_data.columns.tolist()

vif_records = []
X_vif = vif_data.values.astype(float)

for i, col in enumerate(vif_cols):
    try:
        vif_val = variance_inflation_factor(X_vif, i)
    except Exception:
        vif_val = np.nan
    vif_records.append({'feature': col, 'VIF': vif_val})

vif_df = pd.DataFrame(vif_records).set_index('feature').sort_values('VIF', ascending=False)
vif_df['high_VIF'] = vif_df['VIF'] > 10

print(f'Features with VIF > 10: {vif_df["high_VIF"].sum()}')
vif_df.style.format({'VIF': '{:.1f}'}) \
    .apply(lambda col: ['background-color: #ffcccc' if v else '' for v in vif_df['high_VIF']], axis=0, subset=['VIF']) \
    .set_caption('VIF — All Features (high VIF > 10 highlighted)')

## 6. 因子デケイ分析 (Factor Decay)

In [ ]:
HORIZONS = [1, 3, 5, 10, 20]

# Compute forward returns for each horizon
for h in HORIZONS:
    col = f'fwd_ret_{h}d'
    if col not in df.columns:
        df[col] = sym_apply(df, 'close', lambda s, h=h: np.log(s.shift(-h) / s))

def compute_decay_ic(df, feature, horizons, target_prefix='fwd_ret_{}d'):
    results = {}
    for h in horizons:
        target = target_prefix.format(h)
        ic_s = monthly_ic(df, feature, target)
        results[h] = ic_s.mean() if len(ic_s) > 0 else np.nan
    return results

decay_results = {}
for feat in top5:
    decay_results[feat] = compute_decay_ic(df, feat, HORIZONS)

decay_df = pd.DataFrame(decay_results, index=HORIZONS)
print('Factor Decay — Mean Spearman IC by Horizon')
print(decay_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for feat in top5:
    vals = [decay_results[feat][h] for h in HORIZONS]
    ax.plot(HORIZONS, vals, marker='o', label=feat, linewidth=1.8)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Factor Decay — IC vs Forward Return Horizon')
ax.set_xlabel('Forward Return Horizon (days)')
ax.set_ylabel('Mean Spearman IC')
ax.set_xticks(HORIZONS)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. 因子ターンオーバー

In [ ]:
def compute_turnover(df, feature, n_top=20):
    """
    Monthly factor rank correlation between consecutive months.
    Turnover = 1 - mean_consecutive_rank_corr
    """
    tmp = df[['date', 'symbol', feature]].dropna()
    tmp['period'] = tmp['date'].dt.to_period('M')
    
    # For each month, rank across symbols
    monthly_ranks = {}
    for period, grp in tmp.groupby('period'):
        ranked = grp.set_index('symbol')[feature].rank(pct=True)
        monthly_ranks[period] = ranked
    
    periods = sorted(monthly_ranks.keys())
    consec_corrs = []
    for t, t1 in zip(periods[:-1], periods[1:]):
        r0 = monthly_ranks[t]
        r1 = monthly_ranks[t1]
        common = r0.index.intersection(r1.index)
        if len(common) < 5:
            continue
        rho, _ = spearmanr(r0[common], r1[common])
        consec_corrs.append(rho)
    
    if len(consec_corrs) == 0:
        return np.nan
    mean_corr = np.nanmean(consec_corrs)
    return 1.0 - mean_corr

top20_features = ic_summary.head(20).index.tolist()

print('Computing turnover for top 20 features...')
turnover_map = {}
for feat in top20_features:
    turnover_map[feat] = compute_turnover(df, feat)
    print(f'  {feat}: {turnover_map[feat]:.4f}')

print('Done.')

In [ ]:
ic_summary['turnover'] = ic_summary.index.map(turnover_map)

turnover_table = ic_summary.loc[top20_features, ['mean_IC', 'IR', 'IC_pos_ratio', 'turnover']].copy()
print('Factor Summary: IC, IR, IC Positive Ratio, Turnover (top 20 by |mean_IC|)')
turnover_table.style \
    .format({'mean_IC': '{:.4f}', 'IR': '{:.2f}', 'IC_pos_ratio': '{:.1f}%', 'turnover': '{:.4f}'}) \
    .background_gradient(subset=['mean_IC'], cmap='RdBu', vmin=-0.1, vmax=0.1) \
    .background_gradient(subset=['turnover'], cmap='YlOrRd', vmin=0, vmax=1) \
    .set_caption('Factor Summary Table — Top 20 by |mean IC|')

## 8. 因子クロスセクション分散の時系列

In [ ]:
fig, axes = plt.subplots(len(top5), 1, figsize=(14, 3.5 * len(top5)), sharex=False)
if len(top5) == 1:
    axes = [axes]

for ax, feat in zip(axes, top5):
    tmp = df[['date', 'symbol', feat]].dropna()
    # Daily CS std
    daily_cs_std = tmp.groupby('date')[feat].std()
    # Monthly mean of CS std
    daily_cs_std.index = pd.to_datetime(daily_cs_std.index)
    monthly_cs_std = daily_cs_std.resample('ME').mean()
    
    mean_disp = monthly_cs_std.mean()
    ax.plot(monthly_cs_std.index, monthly_cs_std.values, linewidth=1.4, color='steelblue')
    ax.axhline(mean_disp, color='tomato', linewidth=1.2, linestyle='--')
    ax.annotate(f'Mean = {mean_disp:.4f}', xy=(monthly_cs_std.index[-1], mean_disp),
                fontsize=8, color='tomato', va='bottom', ha='right')
    ax.set_title(f'CS Dispersion (monthly std across symbols): {feat}', fontsize=9)
    ax.set_ylabel('CS Std')
    ax.tick_params(axis='x', labelrotation=30, labelsize=8)

plt.suptitle('Factor Cross-Sectional Dispersion — Top 5 Features', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 9. 因子サマリーテーブル

In [ ]:
def compute_decay_half_life(decay_dict, horizons=HORIZONS):
    """
    Horizon at which |IC| drops below 50% of |IC at h=1d|.
    Returns the first horizon where this occurs, or NaN.
    """
    ic_h1 = decay_dict.get(1, np.nan)
    if np.isnan(ic_h1) or ic_h1 == 0:
        return np.nan
    threshold = abs(ic_h1) * 0.5
    for h in sorted(horizons):
        ic_h = decay_dict.get(h, np.nan)
        if not np.isnan(ic_h) and abs(ic_h) < threshold:
            return h
    return max(horizons)  # never dropped below threshold

# Compute decay half-life for top5 (others set to NaN)
decay_half_life_map = {}
for feat in ic_summary.index:
    if feat in decay_results:
        decay_half_life_map[feat] = compute_decay_half_life(decay_results[feat])
    else:
        decay_half_life_map[feat] = np.nan

ic_summary['decay_half_life'] = ic_summary.index.map(decay_half_life_map)

final_cols = ['mean_IC', 'IR', 't_stat', 'IC_pos_ratio', 'decay_half_life', 'turnover']
final_summary = ic_summary[final_cols].sort_values('mean_IC', ascending=False).copy()

print('=== 因子サマリーテーブル (Final Factor Summary) ===')
final_summary.style \
    .format({
        'mean_IC': '{:.4f}',
        'IR': '{:.2f}',
        't_stat': '{:.2f}',
        'IC_pos_ratio': '{:.1f}%',
        'decay_half_life': '{:.0f}d',
        'turnover': '{:.4f}'
    }, na_rep='—') \
    .background_gradient(subset=['mean_IC'], cmap='RdBu', vmin=-0.1, vmax=0.1) \
    .background_gradient(subset=['IR'], cmap='coolwarm', vmin=-2, vmax=2) \
    .background_gradient(subset=['IC_pos_ratio'], cmap='Greens', vmin=40, vmax=80) \
    .background_gradient(subset=['turnover'], cmap='YlOrRd', vmin=0, vmax=1) \
    .set_caption('Factor Summary — Sorted by mean_IC (desc)')

In [ ]:
# Print plain text version for quick reading
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 120)
print(final_summary.to_string())

---
## Conclusion

| 観点 | ポイント |
|---|---|
| **IC** | `mean_IC` と `t_stat` が高い特徴量が予測力の候補 |
| **IR** | `IR > 0.5` が実用的な閾値の目安 |
| **安定性** | 年別 IC ヒートマップ・ローリング IC で確認 |
| **因子相関** | 高相関ペア・高 VIF は冗長 → 次元削減 or 除外を検討 |
| **デケイ** | IC がどの期間まで持続するかでリバランス頻度を決定 |
| **ターンオーバー** | 低い turnover (＝高い rank stability) はトレードコスト小 |